# 01 — Análise Exploratória de Dados (EDA)
**Dataset:** DataCo Smart Supply Chain  
**Objetivo:** Entender a estrutura dos dados, distribuições e principais características logísticas.


In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 53)
pd.set_option('display.float_format', '{:.2f}'.format)

## 1. Carregamento dos Dados

In [2]:
df = pd.read_csv('../data/raw/DataCoSupplyChainDataset.csv', encoding='latin-1')

print(f'Shape: {df.shape}')
print(f'Linhas: {df.shape[0]:,} | Colunas: {df.shape[1]}')
df.head(3)

Shape: (180519, 53)
Linhas: 180,519 | Colunas: 53


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,Customer Country,Customer Email,Customer Fname,Customer Id,Customer Lname,Customer Password,Customer Segment,Customer State,Customer Street,Customer Zipcode,Department Id,Department Name,Latitude,Longitude,Market,Order City,Order Country,Order Customer Id,order date (DateOrders),Order Id,Order Item Cardprod Id,Order Item Discount,Order Item Discount Rate,Order Item Id,Order Item Product Price,Order Item Profit Ratio,Order Item Quantity,Sales,Order Item Total,Order Profit Per Order,Order Region,Order State,Order Status,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.25,314.64,Advance shipping,0,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Cally,20755,Holloway,XXXXXXXXX,Consumer,PR,5365 Noble Nectar Island,725.00,2,Fitness,18.25,-66.04,Pacific Asia,Bekasi,Indonesia,20755,1/31/2018 22:56,77202,1360,13.11,0.04,180517,327.75,0.29,1,327.75,314.64,91.25,Southeast Asia,Java Occidental,COMPLETE,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.09,311.36,Late delivery,1,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Irene,19492,Luna,XXXXXXXXX,Consumer,PR,2679 Rustic Loop,725.00,2,Fitness,18.28,-66.04,Pacific Asia,Bikaner,India,19492,1/13/2018 12:27,75939,1360,16.39,0.05,179254,327.75,-0.80,1,327.75,311.36,-249.09,South Asia,Rajastán,PENDING,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.78,309.72,Shipping on time,0,73,Sporting Goods,San Jose,EE. UU.,XXXXXXXXX,Gillian,19491,Maldonado,XXXXXXXXX,Consumer,CA,8510 Round Bear Gate,95125.00,2,Fitness,37.29,-121.88,Pacific Asia,Bikaner,India,19491,1/13/2018 12:06,75938,1360,18.03,0.06,179253,327.75,-0.80,1,327.75,309.72,-247.78,South Asia,Rajastán,CLOSED,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class


## 2. Tipos e Valores Ausentes

In [3]:
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)

resumo_missing = pd.DataFrame({'missing': missing, 'pct_%': missing_pct})
print('Colunas com valores ausentes:')
print(resumo_missing)

Colunas com valores ausentes:
                     missing  pct_%
Product Description   180519 100.00
Order Zipcode         155679  86.24
Customer Lname             8   0.00
Customer Zipcode           3   0.00


## 3. Distribuição do Status de Entrega

In [4]:
status_counts = df['Delivery Status'].value_counts().reset_index()
status_counts.columns = ['Status', 'Pedidos']
status_counts['Pct'] = (status_counts['Pedidos'] / len(df) * 100).round(1)

cores = {
    'Late delivery': '#e74c3c',
    'Shipping on time': '#2ecc71',
    'Advance shipping': '#3498db',
    'Shipping canceled': '#95a5a6'
}

fig = px.bar(
    status_counts,
    x='Status', y='Pedidos',
    text='Pct',
    color='Status',
    color_discrete_map=cores,
    title='Distribuição do Status de Entrega — 180.519 pedidos',
    labels={'Pedidos': 'Número de Pedidos', 'Status': ''}
)
fig.update_traces(texttemplate='%{text}%', textposition='outside')
fig.update_layout(showlegend=False, height=450)
fig.show()

## 4. Modal de Envio

In [5]:
ship_counts = df['Shipping Mode'].value_counts().reset_index()
ship_counts.columns = ['Modal', 'Pedidos']
ship_counts['Pct'] = (ship_counts['Pedidos'] / len(df) * 100).round(1)

fig = px.pie(
    ship_counts,
    names='Modal', values='Pedidos',
    title='Distribuição por Modal de Envio',
    hole=0.4
)
fig.update_traces(textinfo='label+percent')
fig.update_layout(height=420)
fig.show()

## 5. Top 10 Regiões por Volume de Pedidos

In [6]:
top_regioes = df['Order Region'].value_counts().head(10).reset_index()
top_regioes.columns = ['Região', 'Pedidos']

fig = px.bar(
    top_regioes.sort_values('Pedidos'),
    x='Pedidos', y='Região',
    orientation='h',
    title='Top 10 Regiões por Volume de Pedidos',
    text='Pedidos',
    color='Pedidos',
    color_continuous_scale='Blues'
)
fig.update_traces(texttemplate='%{text:,}', textposition='outside')
fig.update_layout(height=480, coloraxis_showscale=False)
fig.show()

## 6. Top 10 Categorias de Produto

In [7]:
top_cats = df['Category Name'].value_counts().head(10).reset_index()
top_cats.columns = ['Categoria', 'Pedidos']

fig = px.bar(
    top_cats.sort_values('Pedidos'),
    x='Pedidos', y='Categoria',
    orientation='h',
    title='Top 10 Categorias de Produto',
    text='Pedidos',
    color='Pedidos',
    color_continuous_scale='Teal'
)
fig.update_traces(texttemplate='%{text:,}', textposition='outside')
fig.update_layout(height=480, coloraxis_showscale=False)
fig.show()

## 7. Distribuição do Lead Time (Dias Reais de Envio)

In [8]:
fig = px.histogram(
    df,
    x='Days for shipping (real)',
    nbins=20,
    title='Distribuição do Lead Time Real (dias)',
    labels={'Days for shipping (real)': 'Dias de Envio', 'count': 'Pedidos'},
    color_discrete_sequence=['#3498db']
)
fig.update_layout(height=400)
fig.show()

print(f"Média: {df['Days for shipping (real)'].mean():.1f} dias")
print(f"Mediana: {df['Days for shipping (real)'].median():.0f} dias")
print(f"Máx: {df['Days for shipping (real)'].max():.0f} dias")

Média: 3.5 dias
Mediana: 3 dias
Máx: 6 dias


## 8. Conclusões da EDA

- **180.519 pedidos** de e-commerce global, com dados de 2015 a 2018.
- **54,8% das entregas estão em atraso** — pior indicador de performance logística, tema central da análise.
- **Standard Class** domina com 59,7% do volume — modal mais barato mas com maior risco de atraso.
- **Central America e Western Europe** são as regiões com maior volume de pedidos.
- O lead time real médio é de **3–4 dias**, mas há grande variação por modal e região.
- Dataset bem completo: apenas `Customer Zipcode` e `Order Zipcode` têm valores ausentes (<1%).
